In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
source_path ='abfss://raw@dlretailsales.dfs.core.windows.net/incremental/orders/'
schema_path = 'abfss://raw@dlretailsales.dfs.core.windows.net/schemas/orders/'
checkpoint_path = 'abfss://raw@dlretailsales.dfs.core.windows.net/checkpoints/orders'


In [0]:
dbutils.fs.ls(source_path)

In [0]:
orders_schema = StructType([
    StructField('order_id', IntegerType(), True), StructField('customer_id', IntegerType(), True), StructField('store_id', IntegerType(), True), StructField('promotion_id', IntegerType(), True), StructField('order_date', DateType(), True), StructField('created_at', TimestampType(), True), StructField('updated_at', TimestampType(), True), StructField('_rescued_data', StringType(), True), StructField('_ingested_at', TimestampType(), False), StructField('_source_file', StringType(), False)])

orders_stream = (
    spark.readStream.format('cloudFiles')
        .option('cloudFiles.format', 'csv')
        .option('header', 'true')
        .option("cloudFiles.schemaLocation", schema_path)
        .schema(orders_schema)
        .load(source_path)
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_source_file", col('_metadata.file_path'))
)

In [0]:
orders_stream.printSchema()

In [0]:
spark.table('retail_sales_catalog.bronze.orders').printSchema()

In [0]:
spark.sql("""SELECT COUNT(*) AS row_count FROM retail_sales_catalog.bronze.orders""").show()

In [0]:
spark.sql("""
          ALTER TABLE retail_sales_catalog.bronze.orders
          ADD COLUMNS(
              created_at TIMESTAMP,
              updated_at TIMESTAMP  
          )""")

spark.table('retail_sales_catalog.bronze.orders').printSchema()

In [0]:
#deduplicate each microbatch and keep the latest updated_at per order_id

def merge_orders_to_bronze(microbatch_df, batch_id):
    window = (
        Window
        .partitionBy('order_id')
        .orderBy(col('updated_at').desc())
    )

    source_df = (
        microbatch_df
        .withColumn('_rn', row_number().over(window))
        .filter(col('_rn')==1)
        .drop('_rn','_rescued_data')
        .withColumn(
            'order_date', col('order_date').cast('timestamp')
        )
    )

    target = DeltaTable.forName(
        spark, 
        'retail_sales_catalog.bronze.orders'
    )

    (
        target.alias('t')
        .merge(
            source_df.alias('s'),
            't.order_id = s.order_id'
        )
        .whenMatchedUpdateAll(
            condition="""
            t.updated_at IS NULL
            OR s.updated_at>t.updated_at"""
        )
        .whenNotMatchedInsertAll()
        .execute()
    )



In [0]:
spark.sql("""
          SELECT
            COUNT(*) AS total_rows,
            COUNT(created_at) AS rows_with_created_at,
            COUNT(updated_at) AS rows_with_updated_at
            FROM retail_sales_catalog.bronze.orders """).show()

In [0]:
query = (
    orders_stream.writeStream
        .foreachBatch(merge_orders_to_bronze)
        .option('checkpointLocation', checkpoint_path)
        .trigger(availableNow=True)
        .start()
)

query.awaitTermination()

In [0]:
spark.sql("""
SELECT *
FROM retail_sales_catalog.bronze.orders
WHERE order_id > 300000
ORDER BY order_id
""").show(truncate=False)